# Project OverView

The project was developed for Enbridge, a leading Canadian energy infrastructure company with a significant presence in the renewable energy sector. The objective was to build an AI-powered assistant for wind turbine field engineers responsible for inspecting, maintaining, and repairing wind turbines.

The assistant enabled engineers to quickly access critical information such as maintenance manuals, troubleshooting guides, standard operating procedures (SOPs), safety regulations, and historical maintenance records through natural language queries. By leveraging Generative AI and Retrieval-Augmented Generation (RAG), the solution delivered accurate, context-aware responses from the organization's knowledge base, reducing the need to manually search through large volumes of technical documentation.

This improved field productivity, reduced downtime, accelerated issue resolution, and helped engineers make informed maintenance decisions while ensuring compliance with safety standards.



# RAG

## Indexing Stage

### Azure Blog Storage

All company documents (manuals, reports, diagrams, and safety guidelines) are stored in Microsoft Azure Blob Storage, which provides centralized, scalable, and secure storage

### Azure Function

Whenever a new document is uploaded to Azure Blob Storage, an Azure Function is automatically triggered. This function checks the document and adds a task to an Azure Storage Queue. Another Azure Function reads the task from the queue, opens the document, extracts the content, and sends it to the AI pipeline.

### Azure Document Intelligence

Azure Document Intelligence extracts content from PDFs, scanned files, and images. It identifies document structure (headings, paragraphs, tables, lists) and uses OCR(Optical Character Recognition )  to extract text from scanned documents and diagrams.

The output is converted into structured Markdown instead of plain text, preserving headers and document hierarchy for better chunking.

### PyMuPDF 

If the PDF contains images or diagrams, we use PyMuPDF to extract them and save them separately in Azure Blob Storage. We also store metadata, such as the image name and page number, so each image remains linked to the relevant text.

When a user asks a question, the RAG system retrieves both the relevant text and the associated image references. FastAPI then generates a secure SAS URL so the images can be displayed along with the answer, providing a multimodal response.

### Chucking Strategy 

First, the extracted Markdown content is split based on document headings using LangChain's MarkdownHeaderTextSplitter. This preserves the document's logical structure, ensuring that related sections such as maintenance procedures, safety instructions, and alarm descriptions remain together.

Next, the chunks are optimized using token-based chunking. Smaller sections are merged until they reach the desired chunk size, while larger sections are further divided using RecursiveCharacterTextSplitter to stay within the LLM's token limits.

To preserve context, the system applies chunk overlap, where a small portion of the previous chunk is repeated in the next chunk. This helps maintain continuity when information spans multiple chunks.

After the chunks are created, Azure OpenAI extracts structured metadata for each chunk, such as the procedure name, procedure code, alarm code, related figures or tables, and a short summary.

Finally, each chunk, along with its metadata, is stored in a structured JSON format. 

### Azure OpenAI Embedding

In this step, each document chunk is converted into a vector embedding using text-embedding-ada-002. The embedding is a 1536-dimensional vector that captures the semantic meaning of the text.

This allows the system to perform semantic search, where similar meanings can be matched even if the words are different. All document chunks are converted into embeddings and stored.

### Azure AI Search Index

After generating embeddings, we store each document chunk in Azure AI Search along with its metadata and embedding vector. Azure AI Search indexes both the text and the vector, enabling keyword search as well as semantic search. This helps retrieve the most relevant document chunks for the user's query.

## Retrieval Stage

### authentication and authorization 
Every request first goes through authentication to verify the user's identity and authorization to check their permissions.

### Azure API Management (APIM) Rate Limit

We use Azure API Management (APIM) for rate limiting. APIM checks the number of requests a user sends (for example, 10 requests per minute). If the limit is exceeded, it returns an HTTP 429 (Too Many Requests) response without forwarding the request to FastAPI or Azure OpenAI, protecting the application from abuse and reducing unnecessary LLM costs.

### Azure cache Redis (token limit )

We use Azure Cache for Redis to track each user's daily token usage and block requests that exceed the configured limit before they reach the LLM, helping control costs and prevent misuse.

### Azure AI Language 

Azure AI Language is mainly used for PII detection. It identifies sensitive information such as engineer names, email IDs, employee IDs, and other personal data in user queries or documents. We redact this information before sending the request to the LLM, while preserving technical information like turbine IDs, alarm codes, and component names because they are required for accurate retrieval.

### Azure AI Content Safety guardrails

We apply Azure AI Content Safety guardrails at both input and output stages. Before retrieval, we validate and filter user queries to prevent harmful inputs and prompt injection attacks. After Azure OpenAI generates the response, we apply output guardrails to detect sensitive information, harmful content, or policy violations before sending the answer back to the user.

### Pydantic and Regex Validation 

We first use Pydantic to validate the request schema. It checks that all required fields are present. After that, we use Regex validation to verify specific input patterns, such as conversation IDs or allowed characters, and to sanitize the input if necessary. Together, these validations ensure that only valid and well-formed requests enter the RAG pipeline.

### Query rewrite 

we determine whether the query is a new question or a follow-up. If it's a follow-up, we retrieve the relevant conversation history from Azure Cosmos DB. We then combine the current query with the conversation context and use an LLM to rewrite it into a clear, standalone question. The rewritten query is converted into an embedding and sent to Azure AI Search

### Azure AI Search 

In this stage, the system performs hybrid retrieval using Azure AI Search to find the most relevant document chunks.

First, keyword search is performed using BM25, which retrieves chunks containing exact matches of the query terms. This is especially useful for technical queries involving error codes, component IDs, or specific terminology.

At the same time, the query is converted into an embedding, and vector search is performed using KNN (K-Nearest Neighbors). The system retrieves the Top-K nearest document vectors based on cosine similarity, allowing it to find semantically similar content even when different words are used.

The results from both keyword and vector search are then combined using RRF (Reciprocal Rank Fusion), which produces a single ranked list of relevant document chunks.

Next, Semantic Ranker is applied to re-score these retrieved chunks based on their relevance to the user's query. This improves retrieval accuracy by promoting the most relevant chunks to the top.

Finally, the system performs CRAG (Corrective Retrieval-Augmented Generation). It evaluates the quality and confidence of the retrieved chunks. If the retrieved context is insufficient or has a low relevance score, the system can trigger an additional retrieval or query refinement before sending the final context to the LLM. The final Top-K high-quality chunks are then passed to Azure OpenAI to generate an accurate response.


## Generative Stage

### LLM response 

After retrieval, the final Top-K relevant document chunks are combined with the user's query, conversation history, and a carefully designed system prompt. This complete prompt is sent to Azure OpenAI (GPT-4o). The LLM generates a grounded response using only the retrieved context instead of relying on its own knowledge. The system prompt instructs the model to answer only from the provided documents, avoid hallucinations, and clearly state when sufficient information is not available.

### Self RAG
After the response is generated, a Self-RAG evaluation step checks whether the answer is complete, accurate, and well-supported by the retrieved context. If the response quality is below the defined threshold, the system regenerates the answer using the same retrieved context or refined retrieval. To control latency and cost, regeneration is limited to a maximum of two attempts. If the answer is still not satisfactory, the system returns the best available response or informs the user that sufficient information could not be found.

### Azure AI content Safety

the FastAPI backend performs post-processing before returning it to the user. First, Azure AI Content Safety validates the generated output to ensure it does not contain harmful or policy-violating content. 

### Pydantic scheme validation and regrex

we first validate the structured JSON using a Pydantic schema to ensure all required fields and data types are correct. We then apply regex-based sanitization to clean any unwanted formatting or extra text. Finally, returning the playload response to the frontend.


# Security

## Authentication

Our application uses Microsoft Entra ID in a single-tenant configuration, so only employees from our organization can access the AI Assistant. During authentication, the React frontend redirects unauthenticated users to Microsoft Entra ID, where they sign in using their corporate credentials and complete MFA if required. After successful authentication, Microsoft Entra ID issues an ID token for the frontend and an access token (JWT) for the backend. Every API request includes the JWT, and the FastAPI backend validates its signature, issuer, audience, tenant ID, and expiration time.

## Authorization
After authentication, the backend performs authorization using Role-Based Access Control (RBAC). It extracts the user's role or Azure AD group from the JWT and checks whether the user has permission to perform the requested operation. For example, engineers can query the AI Assistant, while administrators can also upload documents and manage the search index. If the user is authorized, the request enters the AI pipeline, where APIM applies rate limiting, Redis checks the token budget and cache, Azure AI Language performs PII detection, Azure AI Search retrieves relevant documents, and Azure OpenAI generates the final response. If authorization fails, the backend immediately returns a 403 Forbidden response.

# Azure Cache for Redis 

Azure Cache for Redis is used as a high-performance in-memory datastore to store temporary data that needs very fast access. It improves application performance, reduces Azure OpenAI costs, and enables state sharing across multiple FastAPI instances.

## Token Budget 

tracks and limits each user's daily LLM token usage by storing the consumed tokens in Redis using a date-based key. After every successful LLM request, it atomically updates the token count using INCRBY to ensure accuracy even with concurrent requests. Before processing a new request, it checks whether the user has exceeded the daily token limit. The Redis key is configured with a 24-hour TTL, so it automatically expires and resets the user's daily token budget without requiring manual cleanup."

## RAG Cache 

we cache the RAG context in Redis using the Cache-Aside pattern so repeated questions can be answered directly from the cache.It generates a unique SHA-256 key based on the user's question and first checks whether the context already exists in the cache. If it's a cache hit, the cached context is returned immediately, avoiding expensive Azure AI Search queries. If it's a cache miss, the service performs a hybrid search, builds the context, stores it in Redis with a configurable TTL(60 min), and returns it. This reduces latency, lowers search costs, and improves response time for repeated or similar queries

## API Rate Limit 

We use a Redis Sorted Set–based sliding-window rate limiter to track requests per user, enforce request limits (10 RPM) within a time window, and return HTTP 429 when the rate limit is exceeded.

### TTL (Time To Live): 

A Redis expiration timer that automatically deletes the daily token key at midnight, resetting the user's token budget for the next day.

## safety

we use Redis SCAN (scan_iter) instead of the KEYS command because KEYS blocks the single-threaded Redis server while searching all matching keys, which can cause latency spikes in production. scan_iter performs a non-blocking, cursor-based scan and retrieves keys in small batches, allowing other Redis operations to continue without interruption. During cache invalidation, the service iterates through the matching keys, deletes them asynchronously, tracks the number of deleted keys, and logs the result. To improve reliability, the deletion logic is wrapped in a try/except block that catches RedisError, logs the failure, and returns the current deletion count instead of crashing the application


# Observability Metrics

We use **Azure Monitor + Application Insights** to continuously monitor the health, performance, reliability, and cost of our RAG chatbot. These metrics provide aggregated insights across all requests and help us create dashboards, alerts, and capacity planning.

## FastAPI Response Time (Latency)

Measures the total time taken by the backend to process a user request.

**Why it matters:** High latency indicates slower responses and impacts the overall user experience.


## Request Count

Measures the total number of API requests received by the application.

**Why it matters:** Helps monitor application traffic, usage patterns, and system load.


## Error Rate (4xx/5xx)

Measures the percentage of failed API requests.

**Why it matters:** A high error rate indicates application issues, service failures, or invalid client requests.

## Azure OpenAI Response Latency

Measures the time taken by Azure OpenAI (GPT-4o) to generate a response.

**Why it matters:** The LLM usually contributes the largest portion of the total response time, making this metric critical for performance monitoring.

## Total Token Usage

Measures the total number of prompt and completion tokens consumed.

**Why it matters:** Token usage directly impacts Azure OpenAI cost and helps optimize prompts and control daily token budgets.


## Azure AI Search Query Latency

Measures the time required to retrieve relevant document chunks from Azure AI Search.

**Why it matters:** Faster retrieval improves the overall response time of the RAG pipeline.


## Semantic Ranker Latency

Measures the time spent reranking retrieved document chunks using Semantic Ranker.

**Why it matters:** Ensures highly relevant documents are ranked first while keeping additional latency under control.


## Redis Cache Hit Ratio

Measures the percentage of requests served directly from Azure Cache for Redis.

**Why it matters:** A high cache hit ratio reduces Azure OpenAI calls, lowers cost, and improves response time.

## Cosmos DB RU Consumption

Measures the Request Units (RU) consumed by Cosmos DB operations.

**Why it matters:** Helps optimize database performance, avoid throttling, and control operational costs.


## Azure Function Execution Time

Measures the time taken by Azure Functions during document ingestion and indexing.

**Why it matters:** Slow execution delays document processing and makes newly uploaded content available later.


## End-to-End Request Duration

Measures the total time taken for a request to travel through the complete RAG pipeline—from FastAPI, Redis, Cosmos DB, Azure AI Search, and Azure OpenAI until the response is returned.

**Why it matters:** Provides an overall view of application performance and helps identify bottlenecks across multiple services.



# RAG evaluation metrics

## Context Precision

Context Precision measures how many of the retrieved documents are actually relevant to the user’s query.
It helps identify whether the system is returning useful information or unnecessary noise.
Higher context precision improves answer quality by ensuring the model focuses only on relevant context.

## Context Recall

Context Recall measures whether the system retrieves all the relevant information needed to answer a query.
It helps identify if important context is missing from the retrieved results.
Higher context recall ensures the model has enough information to generate complete and accurate answers.

## MRR (Mean Reciprocal Rank)

MRR measures how early the first relevant document appears in the ranked retrieval results.
It gives higher scores when the correct document is ranked at the top.
Higher MRR means users (and the LLM) can find useful information quickly.

## Hit rate@K

Hit Rate@K measures whether at least one relevant document appears in the top K retrieved results.
It checks if the system is able to “hit” the correct context within the first K results.
A higher Hit Rate@K means the retrieval system is more likely to provide useful information for answer generation.

## Context Relevancy
Measures how relevant the retrieved chunks are to the query.
Evaluates retrieval quality.

## Faithfulness (Groundedness)

Measures whether all claims in the generated answer are supported by the retrieved context.
Helps detect hallucinations.
High faithfulness means the model is not inventing information.

## Answer Relevancy
Checks if the generated answer directly addresses the user’s question.
Ensures the response stays on-topic.
High relevance means the answer matches the user’s intent. 

## Answer Correctness
Measures how close the generated answer is to the ground-truth answer.
Often evaluated using semantic similarity rather than exact match.
High correctness means the answer is factually accurate.

## Noise Robustness
Evaluates how well the model handles irrelevant or noisy retrieved context.
A robust system still produces correct answers despite distractions.
High robustness means better stability in real-world scenarios.


